[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch06-nn-architectures/03-tuan-tu-va-song-song.ipynb)

# Tuần tự và song song: RNN, ma trận điểm tập trung và bộ nhớ đệm KV

Mạng nơ-ron hồi tiếp (recurrent neural network, RNN) và cơ chế tập trung (attention mechanism) giải cùng một
bài toán, khuôn mẫu trải theo thời gian, bằng hai giả định ngược nhau. RNN nén cả quá khứ vào một vectơ trạng
thái nhỏ, nên bộ nhớ của nó phẳng nhưng các bước phải chạy lần lượt. Cơ chế tập trung cho mọi vị trí nhìn thẳng
vào mọi vị trí khác, nên mọi thứ chạy song song, nhưng số tương tác tăng theo bình phương độ dài chuỗi. Sổ tay
này tính cả hai cái giá đó cho chính xác, rồi đo chúng trên CPU.

**Bạn sẽ làm:**
1. viết RNN của chương bằng tay, kiểm với `nn.RNN`, đếm tham số, phép tính mỗi bước và bộ nhớ trạng thái;
2. đo RNN theo độ dài chuỗi và theo batch, để thấy đường tới hạn (critical path) nằm ở đâu;
3. tính chính xác bộ nhớ và FLOP của ma trận điểm tập trung, làm lại Napkin Math 1.1, và đo thời gian;
4. viết cơ chế tập trung theo kiểu chia khối (tiling) với softmax trực tuyến, kiểm rằng nó cho đúng kết quả;
5. dựng một khối transformer bằng tay và tìm độ dài chuỗi mà ở đó phần tập trung chiếm phần lớn phép tính;
6. tính bức tường băng thông khi sinh token và bộ nhớ đệm KV (KV cache) của chương.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]
FP32, FP16 = 4, 2


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f() (giây), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


def truc_y_log(ax):
    """Trục y log với nhãn số thường (1, 2, 5, 10, ...) thay vì luỹ thừa."""
    from matplotlib.ticker import FuncFormatter, LogLocator
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:,.6g}"))


print("sẵn sàng")

## 1. Một RNN, viết bằng tay

Phương trình 6 cập nhật trạng thái ẩn từ đầu vào hiện tại và trạng thái trước đó:

$$h_t = \tanh(W_{hh} h_{t-1} + W_{hx} x_t + b_h).$$

Chương lấy ví dụ mỗi từ là một vectơ 100 chiều và trạng thái ẩn có 128 chiều. Ô dưới viết đúng vòng lặp đó, so
với `nn.RNN` của PyTorch, và đếm tham số. (PyTorch giữ hai vectơ bias, một cho mỗi phép nhân, nên nó có thêm 128
tham số so với phương trình 6; tổng của chúng đóng vai $b_h$.)

In [ ]:
D_VAO = sach(100, nguon="100 chiều · 128 chiều")
D_AN = sach(128, nguon="100 chiều · 128 chiều")
dat_hat_giong()
rnn = nn.RNN(D_VAO, D_AN)
W_hx, W_hh = rnn.weight_ih_l0.detach(), rnn.weight_hh_l0.detach()
b_h = (rnn.bias_ih_l0 + rnn.bias_hh_l0).detach()


def rnn_bang_tay(x):
    """x: (S, D_VAO). Trả về mọi trạng thái ẩn, (S, D_AN)."""
    h = torch.zeros(D_AN)
    cac_h = []
    for x_t in x:                                  # bước t phải đợi bước t − 1
        h = torch.tanh(W_hh @ h + W_hx @ x_t + b_h)
        cac_h.append(h)
    return torch.stack(cac_h)


x_chuoi = torch.randn(50, D_VAO)
with torch.no_grad():
    y_pt, _ = rnn(x_chuoi[:, None, :])
assert torch.allclose(rnn_bang_tay(x_chuoi), y_pt[:, 0], atol=1e-5)
print("vòng lặp bằng tay khớp nn.RNN trên 50 bước")
print(f"tham số: W_hh {W_hh.numel():,} + W_hx {W_hx.numel():,} + bias = {sum(p.numel() for p in rnn.parameters()):,}")
theo_sach("hình dạng phép nhân hồi tiếp", W_hh.shape[0], sach=128, nguon="128×128 · 100×128")
theo_sach("hình dạng phép chiếu đầu vào (số chiều vào)", W_hx.shape[1], sach=100, nguon="128×128 · 100×128")
mac_buoc = W_hh.numel() + W_hx.numel()
print(f"mỗi bước: {mac_buoc:,} MAC, trên {W_hh.numel() + W_hx.numel():,} trọng số, tức mỗi trọng số dùng đúng 1 lần")

Mỗi trọng số được dùng đúng một lần mỗi bước, và mỗi bước là một phép nhân ma trận với **vectơ**. Ở batch 1,
đó là tải công việc có mật độ tính toán của tầng dày đặc trong sổ tay 01, khoảng 0.5 FLOP/byte nếu trọng số phải
đọc lại từ bộ nhớ.

### Bộ nhớ trạng thái: phẳng theo độ dài chuỗi

Khi suy luận, RNN chỉ cần giữ trạng thái ẩn hiện tại. Chương lấy ví dụ một trạng thái 512 chiều.

**Dự đoán:** trạng thái đó chiếm bao nhiêu byte ở FP32, với chuỗi 10 bước? Với chuỗi 10,000 bước?

In [ ]:
D_TRANG_THAI = sach(512, nguon="2 KB · trạng thái 512 chiều")
S_NGAN = sach(10, trich="whether the sequence length is 10 or 10,000. This")
S_DAI_RNN = sach(10_000, trich="whether the sequence length is 10 or 10,000. This")
dat_hat_giong()
rnn_512 = nn.RNN(D_TRANG_THAI, D_TRANG_THAI)
with torch.no_grad():
    for S in (S_NGAN, S_DAI_RNN):
        _, h_cuoi = rnn_512(torch.randn(S, 1, D_TRANG_THAI))
        print(f"chuỗi {S:>6,} bước: trạng thái giữ lại {h_cuoi.numel():,} giá trị = {h_cuoi.numel() * FP32:,} byte")
theo_sach("trạng thái 512 chiều ở FP32 (KB, 1 KB = 1,024 byte)", D_TRANG_THAI * FP32 / 1024, sach=2,
          nguon="2 KB · trạng thái 512 chiều")
print(f"nhưng khi huấn luyện, lan truyền ngược theo thời gian giữ mọi trạng thái: {S_DAI_RNN:,} × "
      f"{D_TRANG_THAI} × 4 byte = {S_DAI_RNN * D_TRANG_THAI * FP32 / 1e6:.1f} MB cho một chuỗi, một tầng")

## 2. Đường tới hạn: các bước không chạy song song được

Bước $t$ không bắt đầu được khi bước $t - 1$ chưa xong. Chương nói: một văn bản 1,000 từ là 1,000 phép nhân
ma trận với vectơ phụ thuộc nhau, và phần cứng thêm vào chỉ làm từng phép nhanh hơn chứ không chạy chúng cùng
lúc được. Chiều song song duy nhất còn lại là batch.

Nhưng không phải mọi thứ trong RNN đều tuần tự. Phép chiếu đầu vào $W_{hx} x_t$ không phụ thuộc $h$, nên có
thể tính cho cả chuỗi bằng **một** phép nhân ma trận trước khi vòng lặp bắt đầu. Chỉ $W_{hh} h_{t-1}$ là nằm
trên đường tới hạn.

**Dự đoán:** khi chuỗi dài gấp đôi, thời gian của RNN tăng bao nhiêu? Còn khi batch lớn gấp 8?

In [ ]:
BUOC_PHU_THUOC = sach(1000, trich="the system must execute 1,000 dependent matrix-vector multiplications")
cac_S = [128, 256, 512, 1024, 2048]
t_S = []
for S in cac_S:
    x = torch.randn(S, 1, D_VAO)
    with torch.no_grad():
        t_S.append(thoi_gian(lambda: rnn(x), lap=3))
for S, t in zip(cac_S, t_S):
    do_tren_may(f"nn.RNN, chuỗi {S} bước, batch 1", 1000 * t, "ms")

x_1000 = torch.randn(BUOC_PHU_THUOC, D_VAO)
with torch.no_grad():
    t_chieu = thoi_gian(lambda: x_1000 @ W_hx.T)          # cả chuỗi, một GEMM
    P = x_1000 @ W_hx.T + b_h                              # phần không phụ thuộc h, tính trước

    def chuoi_hoi_tiep():
        h = torch.zeros(D_AN)
        for t in range(BUOC_PHU_THUOC):                    # phần nằm trên đường tới hạn
            h = torch.tanh(P[t] + W_hh @ h)
        return h

    t_chuoi = thoi_gian(chuoi_hoi_tiep, lap=3)
do_tren_may("phép chiếu đầu vào cho cả 1,000 bước, một GEMM", 1000 * t_chieu, "ms")
do_tren_may("1,000 bước hồi tiếp nối nhau, vòng lặp Python", 1000 * t_chuoi, "ms")

cac_B = [1, 8, 64]
for B in cac_B:
    x = torch.randn(512, B, D_VAO)
    with torch.no_grad():
        t = thoi_gian(lambda: rnn(x), lap=3)
    do_tren_may(f"nn.RNN, 512 bước, batch {B}: FLOP/s đạt được", 2 * 512 * B * mac_buoc / t / 1e9, "GFLOP/s")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.plot(cac_S, [1000 * t for t in t_S], "o-", color=MAU[0], label="nn.RNN, batch 1")
ax.plot(cac_S, [1000 * t_S[0] * S / cac_S[0] for S in cac_S], "--", color="0.5",
        label="tỉ lệ thuận với độ dài chuỗi")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(cac_S, [str(s) for s in cac_S])
ax.minorticks_off()
truc_y_log(ax)
ax.set_xlabel("độ dài chuỗi (số bước)")
ax.set_ylabel("thời gian (ms)")
ax.set_title("RNN 100 → 128: thời gian theo độ dài chuỗi (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Đường đứt nét là "thời gian tỉ lệ thuận với số bước", điều mà một chuỗi phụ thuộc buộc phải có: không có cách
nào xong bước thứ 2,048 trước khi xong bước thứ 2,047. Thêm lõi, thêm băng thông chỉ làm từng bước ngắn lại.
Vòng lặp Python ở trên còn cho thấy một cái giá nữa: mỗi bước là vài lần gọi kernel nhỏ, và phụ trội cố định
của mỗi lần gọi không gộp được vào bước khác, vì bước sau chưa có dữ liệu để chạy. So nó với một GEMM cho cả
phép chiếu đầu vào.
Batch thì khác: các chuỗi khác nhau độc lập với nhau, nên gom batch biến phép nhân ma trận với vectơ thành phép
nhân ma trận với ma trận. So FLOP/s đạt được ở ba batch in ra ở trên: đó là đòn bẩy duy nhất RNN có để dùng phần
cứng nhiều hơn.

## 3. Ma trận điểm tập trung: S × S

Cơ chế tập trung tính $\text{softmax}(QK^\top / \sqrt{d_k})\,V$. Tích $QK^\top$ là một ma trận $S \times S$: mỗi
vị trí một hàng, mỗi vị trí một cột. Một hiện thực ngây thơ ghi cả ma trận đó ra bộ nhớ. Chương tính cỡ của nó
cho chuỗi 4,096 token ở 16 bit.

**Dự đoán:** gấp đôi độ dài chuỗi thì ma trận đó lớn gấp mấy? Từ 512 lên 4,096 thì sao?

In [ ]:
S_4096 = sach(4096, nguon="4.096 token · 33,6 MB")
so_diem = S_4096 * S_4096
theo_sach("số điểm, một tầng, một đầu", so_diem, sach=16.8e6, trich="about 16.8M scores at 2 bytes each")
theo_sach("bộ nhớ điểm FP16, một tầng, một đầu (MB)", so_diem * FP16 / 1e6, sach=33.6, nguon="4.096 token · 33,6 MB")

S_BERT = sach(512, nguon="512 → 4.096 = 64×")
theo_sach("từ 512 lên 4,096 token (lần)", (S_4096 / S_BERT) ** 2, sach=64, nguon="512 → 4.096 = 64×")
S_2048 = sach(2048, nguon="2.048 so với 512 = 16×")
theo_sach("2,048 so với 512 token (lần)", (S_2048 / S_BERT) ** 2, sach=16, nguon="2.048 so với 512 = 16×")

# Napkin Math 1.1: ngữ cảnh 100,000 token, 12 đầu, 32 tầng giữ lại khi huấn luyện.
S_DAI = sach(100_000, nguon="100.000 × 100.000 × 12 đầu")
DAU = sach(12, nguon="100.000 × 100.000 × 12 đầu")
TANG = sach(32, nguon="240 GB × 32 tầng = 7.680 GB")
phan_tu = S_DAI * S_DAI * DAU
theo_sach("số phần tử, một tầng", phan_tu, sach=1.2e11, nguon="1,2 × 10¹¹")
theo_sach("bộ nhớ FP16, một tầng (GB)", phan_tu * FP16 / 1e9, sach=240, nguon="1,2 × 10¹¹ · 240 GB")
theo_sach("32 tầng giữ lại cho lượt ngược (GB)", phan_tu * FP16 * TANG / 1e9, sach=7680, nguon="240 GB × 32 tầng = 7.680 GB")

Ma trận điểm tăng theo bình phương: gấp đôi chuỗi là gấp bốn. Đồ thị dưới đặt nó cạnh hai loại trạng thái khác
để bạn thấy ba cách lớn lên của ba kiến trúc: trạng thái ẩn của RNN (phẳng), bộ nhớ đệm khoá và giá trị của một
đầu (tuyến tính, phần 6 sẽ nói kỹ), và ma trận điểm của một đầu (bậc hai). Các vạch dọc là cửa sổ ngữ cảnh mà
chương kể: khoảng 512 tới 2K token của các mô hình đầu tiên, rồi 128K, 200K và hơn 1 triệu token.

In [ ]:
D_DAU = sach(128, trich="128-dimensional heads")
truc_S = np.logspace(7, 20, 27, base=2)
fig, ax = plt.subplots(figsize=(7.8, 4.6))
ax.loglog(truc_S, [D_TRANG_THAI * FP32] * len(truc_S), color=MAU[0], label="RNN: trạng thái ẩn 512 chiều, FP32")
ax.loglog(truc_S, 2 * truc_S * D_DAU * FP16, color=MAU[2], label="bộ nhớ đệm KV: một tầng, một đầu 128 chiều, FP16")
ax.loglog(truc_S, truc_S**2 * FP16, color=MAU[1], label="ma trận điểm: một tầng, một đầu, FP16")
for S, chu in ((sach(512, nguon="512–2K · 128K"), "512"), (sach(2e3, nguon="512–2K · 128K"), "2K"),
               (sach(128e3, nguon="512–2K · 128K"), "128K"), (sach(200e3, nguon="128K · 200K"), "200K"),
               (sach(1e6, nguon="200K · 1 triệu+"), "1 triệu")):
    ax.axvline(S, color="0.8", lw=0.8, zorder=0)
    ax.text(S * 1.05, 1.5e12, chu, fontsize=7.5, color="0.4", va="top")
ax.annotate("33.6 MB ở 4,096 token", (S_4096, so_diem * FP16), xytext=(-150, 22), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.annotate("2 KB", (S_4096, D_TRANG_THAI * FP32), xytext=(0, 8), textcoords="offset points", fontsize=8)
for y, chu in ((1e3, "1 KB"), (1e6, "1 MB"), (1e9, "1 GB"), (1e12, "1 TB")):
    ax.axhline(y, color="0.92", lw=0.6, zorder=0)
ax.set_yticks([1e3, 1e6, 1e9, 1e12], ["1 KB", "1 MB", "1 GB", "1 TB"])
ax.set_ylim(5e2, 3e12)
ax.set_xlabel("độ dài chuỗi (token, thang log)")
ax.set_ylabel("byte (thang log)")
ax.set_title("Ba cách trạng thái lớn lên theo độ dài chuỗi")
ax.legend(loc="lower right", fontsize=7.5, bbox_to_anchor=(1, 0.08))
plt.tight_layout()
plt.show()
for S in (128e3, 1e6):
    print(f"ma trận điểm, một tầng một đầu, ở {S:,.0f} token: {S * S * FP16 / 1e9:,.0f} GB")

### FLOP và thời gian

Chương đếm $S \times S \times d$ phép nhân cộng dồn cho $QK^\top$, và bằng chừng đó cho phép nhân trọng số tập
trung với $V$. Tổng là $2 S^2 d$ MAC, tức $4 S^2 d$ FLOP, chưa kể các phép chiếu và softmax. Ô dưới đo một đầu
với $d = 64$ trên CPU này, ở năm độ dài chuỗi.

In [ ]:
d_dau = 64
cac_S_do = [256, 512, 1024, 2048, 4096]
t_att = []
for S in cac_S_do:
    q, k, v = torch.randn(S, d_dau), torch.randn(S, d_dau), torch.randn(S, d_dau)
    t_att.append(thoi_gian(lambda: torch.softmax(q @ k.T / math.sqrt(d_dau), dim=-1) @ v, lap=3))
for S, t in zip(cac_S_do, t_att):
    do_tren_may(f"tập trung một đầu, S = {S}: thời gian", 1000 * t, "ms")
    do_tren_may(f"tập trung một đầu, S = {S}: FLOP/s đạt được", 4 * S * S * d_dau / t / 1e9, "GFLOP/s")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.plot(cac_S_do, [1000 * t for t in t_att], "o-", color=MAU[1], label="đo được")
ax.plot(cac_S_do, [1000 * t_att[1] * (S / cac_S_do[1]) ** 2 for S in cac_S_do], "--", color="0.5",
        label="tỉ lệ với S² (độ dốc 2)")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(cac_S_do, [str(s) for s in cac_S_do])
ax.minorticks_off()
truc_y_log(ax)
ax.set_xlabel("độ dài chuỗi S")
ax.set_ylabel("thời gian (ms)")
ax.set_title("Tập trung một đầu, d = 64, FP32 (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Trên thang log, một đường tỉ lệ với $S^2$ có độ dốc 2. Nếu điểm đo nằm trên đường đứt nét ở chuỗi dài, thời gian
đang đi theo số phép tính; nếu nó vọt lên trên, thường là vì ma trận điểm không còn vừa trong cache và mỗi lượt
phải ra bộ nhớ chính. Đường RNN ở phần 2 có độ dốc 1, nhưng mỗi bước của nó phải đợi bước trước. Ở đây, mọi hàng
của $QK^\top$ độc lập với nhau, nên cả phép tính là một GEMM lớn. Đó là toàn bộ chuyện song song hoá: tập trung
làm nhiều phép tính hơn RNN, nhưng không có phép tính nào phải đợi phép tính nào.

## 4. Chia khối: giữ phép tính, bỏ ma trận

Chương nói phép tính $S^2$ của tập trung dày đặc là không tránh được, còn việc lưu ma trận $S^2$ thì có thể tránh.
Cách làm là chia các khoá thành từng khối, tính điểm cho một khối, rồi cộng dồn vào kết quả bằng softmax trực
tuyến (online softmax): giữ cho mỗi hàng giá trị lớn nhất và tổng mũ đã gặp, và chỉnh lại kết quả cũ mỗi khi giá
trị lớn nhất tăng. Mỗi lúc chỉ có một khối $S \times B_k$ điểm tồn tại.

In [ ]:
S, Bk = 2048, 256
dat_hat_giong()
q, k, v = torch.randn(S, d_dau), torch.randn(S, d_dau), torch.randn(S, d_dau)


def tap_trung_day_du(q, k, v):
    return torch.softmax(q @ k.T / math.sqrt(d_dau), dim=-1) @ v, q.shape[0] * k.shape[0]


def tap_trung_chia_khoi(q, k, v, Bk):
    m = torch.full((q.shape[0], 1), -math.inf)   # giá trị lớn nhất đã gặp, mỗi hàng
    l = torch.zeros(q.shape[0], 1)               # tổng exp(điểm − m) đã gặp
    o = torch.zeros_like(q)
    dinh = 0
    for j in range(0, k.shape[0], Bk):
        diem = q @ k[j:j + Bk].T / math.sqrt(d_dau)  # chỉ một khối S × Bk
        dinh = max(dinh, diem.numel())
        m_moi = torch.maximum(m, diem.max(dim=1, keepdim=True).values)
        p = torch.exp(diem - m_moi)
        he_so = torch.exp(m - m_moi)                 # chỉnh lại phần đã cộng dồn
        l = l * he_so + p.sum(dim=1, keepdim=True)
        o = o * he_so + p @ v[j:j + Bk]
        m = m_moi
    return o / l, dinh


o_day_du, dinh_day_du = tap_trung_day_du(q, k, v)
o_khoi, dinh_khoi = tap_trung_chia_khoi(q, k, v, Bk)
assert torch.allclose(o_day_du, o_khoi, atol=1e-5)
print(f"chia khối khớp bản đầy đủ: lệch lớn nhất {(o_day_du - o_khoi).abs().max():.1e}")
print(f"MAC: cả hai đều {2 * S * S * d_dau:,} (QKᵀ và nhân với V)")
print(f"điểm tồn tại cùng lúc: đầy đủ {dinh_day_du:,} ({dinh_day_du * FP32 / 1e6:.1f} MB FP32), "
      f"chia khối {dinh_khoi:,} ({dinh_khoi * FP32 / 1e6:.1f} MB) = 1/{dinh_day_du // dinh_khoi}")
do_tren_may("đầy đủ, thời gian", 1000 * thoi_gian(lambda: tap_trung_day_du(q, k, v), lap=3), "ms")
do_tren_may("chia khối, thời gian", 1000 * thoi_gian(lambda: tap_trung_chia_khoi(q, k, v, Bk), lap=3), "ms")

Kết quả giống nhau tới sai số làm tròn, số phép tính giống hệt, còn bộ nhớ cho điểm giảm theo tỉ lệ $S / B_k$. Đó
là ý tưởng của FlashAttention: trên GPU, khối điểm nằm trong SRAM trên chip và không bao giờ ra HBM. Trên CPU này
hai bản có thể nhanh chậm khác nhau theo cách khác, vì vòng lặp Python và cache của CPU không phải SRAM của GPU;
điều sổ tay khẳng định chỉ là phép tính giữ nguyên và ma trận đầy đủ không còn.

## 5. Một khối transformer, đếm từng phần

Một khối transformer gồm tự tập trung (self-attention) nhiều đầu, một mạng truyền thẳng hai tầng, hai lớp chuẩn
hoá theo tầng và hai kết nối tắt. Ô dưới dựng nó từ các `nn.Linear` và `nn.LayerNorm`, kiểm số tham số với
`nn.TransformerEncoderLayer` của PyTorch, và kiểm rằng hai bản cho cùng đầu ra khi dùng chung trọng số.

In [ ]:
d_mo_hinh, so_dau, d_ff = 512, 8, 2048


class KhoiTransformer(nn.Module):
    def __init__(self, d, dau, d_ff):
        super().__init__()
        self.dau, self.d_dau = dau, d // dau
        self.qkv = nn.Linear(d, 3 * d)      # W_Q, W_K, W_V gộp làm một
        self.ra = nn.Linear(d, d)           # W_O
        self.ff1, self.ff2 = nn.Linear(d, d_ff), nn.Linear(d_ff, d)
        self.chuan1, self.chuan2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x):                   # x: (S, d)
        S = x.shape[0]
        q, k, v = self.qkv(x).view(S, 3, self.dau, self.d_dau).permute(1, 2, 0, 3)
        diem = torch.softmax(q @ k.transpose(-1, -2) / math.sqrt(self.d_dau), dim=-1)  # (dau, S, S)
        a = (diem @ v).transpose(0, 1).reshape(S, -1)
        x = self.chuan1(x + self.ra(a))
        return self.chuan2(x + self.ff2(F.relu(self.ff1(x))))


dat_hat_giong()
khoi = KhoiTransformer(d_mo_hinh, so_dau, d_ff).eval()
chuan = nn.TransformerEncoderLayer(d_mo_hinh, so_dau, d_ff, dropout=0.0).eval()
so_tham_so = sum(p.numel() for p in khoi.parameters())
assert so_tham_so == sum(p.numel() for p in chuan.parameters())
with torch.no_grad():                       # chép trọng số sang bản của PyTorch rồi so đầu ra
    chuan.self_attn.in_proj_weight.copy_(khoi.qkv.weight)
    chuan.self_attn.in_proj_bias.copy_(khoi.qkv.bias)
    chuan.self_attn.out_proj.load_state_dict(khoi.ra.state_dict())
    chuan.linear1.load_state_dict(khoi.ff1.state_dict())
    chuan.linear2.load_state_dict(khoi.ff2.state_dict())
    chuan.norm1.load_state_dict(khoi.chuan1.state_dict())
    chuan.norm2.load_state_dict(khoi.chuan2.state_dict())
    x = torch.randn(64, d_mo_hinh)
    lech = (khoi(x) - chuan(x[:, None, :])[:, 0]).abs().max()
assert lech < 1e-4
print(f"khối tự dựng: {so_tham_so:,} tham số, đúng bằng nn.TransformerEncoderLayer; lệch đầu ra {lech:.1e}")

Giờ đếm phép tính cho mỗi token. Bốn phép chiếu $Q, K, V, O$ cần $4d^2$ MAC, mạng truyền thẳng cần $2 d \cdot
d_{ff}$, và cả hai không phụ thuộc độ dài chuỗi. Phần tập trung cần $2 S d$ MAC cho mỗi token (mỗi token so với
$S$ khoá, rồi cộng $S$ giá trị). Với $d_{ff} = 4d$, phần tuyến tính là $12 d^2$, nên phần tập trung vượt nó khi
$2Sd > 12d^2$, tức khi $S > 6d$. Đây là phép tính của riêng sổ tay này, không phải của sách.

**Dự đoán:** với $d = 512$, phần tập trung bắt đầu chiếm đa số phép tính từ độ dài chuỗi nào?

In [ ]:
def mac_moi_token(S, d=d_mo_hinh, d_ff=d_ff):
    return {"chiếu Q, K, V, O": 4 * d * d, "truyền thẳng": 2 * d * d_ff, "tập trung (S²)": 2 * S * d}


cac_S_khoi = [2**k for k in range(6, 16)]
phan_tap_trung = [mac_moi_token(S)["tập trung (S²)"] / sum(mac_moi_token(S).values()) for S in cac_S_khoi]
diem_vuot = 6 * d_mo_hinh
assert mac_moi_token(diem_vuot)["tập trung (S²)"] == mac_moi_token(diem_vuot)["chiếu Q, K, V, O"] \
    + mac_moi_token(diem_vuot)["truyền thẳng"]
fig, ax = plt.subplots(figsize=(6.8, 3.6))
ax.semilogx(cac_S_khoi, [100 * p for p in phan_tap_trung], "o-", color=MAU[1], base=2)
ax.axvline(diem_vuot, color="0.6", lw=0.8, ls="--")
ax.text(diem_vuot * 1.08, 12, f"S = 6d = {diem_vuot:,}", fontsize=8, color="0.35")
ax.axhline(50, color="0.85", lw=0.8)
ax.set_xticks(cac_S_khoi[::2], [str(s) for s in cac_S_khoi[::2]])
ax.set_ylim(0, 100)
ax.set_xlabel("độ dài chuỗi S")
ax.set_ylabel("phần MAC của tập trung (%)")
ax.set_title("Khối transformer d = 512, d_ff = 2,048: phần tập trung trong mỗi token")
plt.tight_layout()
plt.show()
for S in (512, 4096, 32_768):
    p = mac_moi_token(S)
    print(f"S = {S:>6,}: {sum(p.values()) / 1e6:6.2f} triệu MAC mỗi token, tập trung chiếm "
          f"{100 * p['tập trung (S²)'] / sum(p.values()):.0f} %")

Ở chuỗi ngắn, một khối transformer chủ yếu là phép nhân ma trận với trọng số, đúng loại việc mà sổ tay 01 đếm. Ở
chuỗi dài, phần bậc hai lấn át. Điều này giải thích vì sao cùng một mô hình có thể nghẽn ở chỗ khác nhau tuỳ độ dài
ngữ cảnh.

## 6. Sinh token: bức tường băng thông

Khi huấn luyện và trong giai đoạn xử lý prompt (prefill), mọi token đi qua cùng lúc. Khi sinh token theo kiểu tự hồi
quy (autoregressive), mô hình sinh **một** token mỗi lượt, nên mỗi tầng dày đặc lại là một phép nhân ma trận với
vectơ: mọi trọng số được đọc, mỗi trọng số dùng cho đúng hai phép tính. Bảng của chương cho GPT-2 XL:

In [ ]:
GPT_GB = sach(6, nguon="6 GB · 3 GFLOP/token · 0,5–1 FLOP/byte")
GPT_GFLOP = sach(3, nguon="6 GB · 3 GFLOP/token")
theo_sach("mật độ tính toán, trọng số FP32", GPT_GFLOP / GPT_GB, sach=0.5, nguon="3 GFLOP/token · 0,5–1 FLOP/byte")
theo_sach("mật độ tính toán, trọng số FP16 (một nửa số byte)", GPT_GFLOP / (GPT_GB / 2), sach=1,
          nguon="3 GFLOP/token · 0,5–1 FLOP/byte")

LLM_GB = sach(60, nguon="60 GB · 2 TB/s · 33,3 token/giây")
BANG_THONG = sach(2, nguon="60 GB · 2 TB/s") * 1e12
theo_sach("trần token/giây khi mỗi token đọc lại toàn bộ trọng số", BANG_THONG / (LLM_GB * 1e9), sach=33.3,
          nguon="2 TB/s · 33,3 token/giây")
print("trần này không phụ thuộc phần cứng tính nhanh cỡ nào: nó chỉ là băng thông chia cho số byte mỗi token")
TY_70 = sach(70, trich="a 70-billion-parameter model stores 140 GB of FP16 weights") * 1e9
theo_sach("mô hình 70 tỷ tham số, trọng số FP16 (GB)", TY_70 * FP16 / 1e9, sach=140,
          trich="a 70-billion-parameter model stores 140 GB of FP16 weights")

Gom batch là đòn bẩy: $B$ yêu cầu dùng chung một lần đọc trọng số, nên token mỗi giây tăng gần theo $B$ cho tới khi
phép tính, chứ không phải băng thông, trở thành giới hạn. Ô dưới đo đúng hiện tượng đó trên CPU, với một ma trận
trọng số 4,096 × 4,096 FP32 (64 MB, lớn hơn cache của hầu hết CPU) nhân với $B$ vectơ.

**Dự đoán:** từ batch 1 lên batch 64, token mỗi giây tăng bao nhiêu lần?

In [ ]:
dat_hat_giong()
W_lon = torch.randn(4096, 4096)
cac_B = [1, 2, 4, 8, 16, 32, 64, 128, 256]
t_B = []
for B in cac_B:
    X = torch.randn(B, 4096)
    t_B.append(thoi_gian(lambda: X @ W_lon.T, lap=5))
do_tren_may("batch 1: băng thông đọc trọng số đạt được", W_lon.numel() * FP32 / t_B[0] / 1e9, "GB/s")
for B, t in zip(cac_B, t_B):
    do_tren_may(f"batch {B}: vectơ mỗi giây", B / t, "")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.plot(cac_B, [B / t for B, t in zip(cac_B, t_B)], "o-", color=MAU[0], label="đo được")
ax.plot(cac_B, [B / t_B[0] for B in cac_B], "--", color="0.5", label="nếu thời gian giữ nguyên như ở batch 1")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(cac_B, [str(b) for b in cac_B])
ax.minorticks_off()
truc_y_log(ax)
ax.set_xlabel("batch (số yêu cầu dùng chung một lần đọc trọng số)")
ax.set_ylabel("vectơ mỗi giây")
ax.set_title("Ma trận 4,096 × 4,096 FP32 nhân với B vectơ (đo trên máy này)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Đường đứt nét là trường hợp lý tưởng của một tải công việc nghẽn ở bộ nhớ: đọc trọng số mất cùng thời gian dù
dùng cho một vectơ hay cho nhiều. Khoảng cách giữa điểm đo và đường đó cho biết lúc nào phép tính bắt đầu chiếm
phần đáng kể. Ở batch rất nhỏ, đường đo có thể không tăng đều, vì thư viện có thể đổi giữa kernel nhân ma trận
với vectơ và kernel nhân ma trận với ma trận. Trên một CPU một luồng, chỗ đó đến sớm hơn nhiều so với trên GPU, nơi phần tính toán mạnh hơn
băng thông hàng trăm lần.

### Bộ nhớ đệm KV

Để không tính lại khoá và giá trị của mọi token cũ ở mỗi bước, hệ thống giữ chúng trong bộ nhớ đệm KV, lớn lên
tuyến tính theo độ dài chuỗi. Chương tính cho ví dụ 32 tầng, 32 đầu, đầu 128 chiều, chuỗi 2,048 token, FP16.

In [ ]:
TANG_KV = sach(32, nguon="32 tầng · 32 đầu")
DAU_KV = sach(32, nguon="32 tầng · 32 đầu")
S_KV = sach(2048, trich="2,048-token sequences")
kv_byte = TANG_KV * 2 * DAU_KV * S_KV * D_DAU * FP16       # 2: một cho khoá, một cho giá trị
theo_sach("bộ nhớ đệm KV mỗi yêu cầu (GB)", kv_byte / 1e9, sach=1, trich="1 GB of KV cache", tol=0.1)
print(f"chính xác: {kv_byte:,} byte = {kv_byte / 1e9:.2f} GB = đúng {kv_byte / 2**30:.0f} GiB")
THIET_BI = sach(80, trich="Those values do not exhaust a 80 GB device")
for nguoi in (2, 4):
    print(f"{nguoi} người dùng cùng lúc: {nguoi * kv_byte / 1e9:.1f} GB, {100 * nguoi * kv_byte / (THIET_BI * 1e9):.1f} % "
          f"của thiết bị {THIET_BI} GB, trước mọi bộ đệm khác")
print(f"một yêu cầu ở 128K token: {kv_byte * 128_000 / S_KV / 1e9:.0f} GB")

Bộ nhớ đệm KV ra đúng $2^{30}$ byte: chương làm tròn thành "khoảng 1 GB", và theo đơn vị thập phân con số chính xác
là 1.07 GB. Hai tới bốn người dùng chưa lấp đầy một thiết bị 80 GB, nhưng phần đó lớn lên tuyến tính theo cả độ dài
ngữ cảnh lẫn số người dùng, và ở ngữ cảnh 128K token thì một yêu cầu đã cần hàng chục GB.

## Thử thêm

1. Ở phần 2, thay `nn.RNN` bằng `nn.LSTM(D_VAO, D_AN)`. Số tham số gấp mấy lần, và thời gian mỗi bước đổi ra sao?
   LSTM có giải được đường tới hạn không?
2. Ở phần 4, đổi `Bk` thành 64 rồi 1024. Bộ nhớ đỉnh và thời gian đổi thế nào? Kết quả có còn khớp bản đầy đủ?
3. Ở phần 5, đặt `d_ff = 8 * d_mo_hinh`. Điểm vượt $S = 6d$ dời về đâu? Viết lại điều kiện cho $d_{ff}$ bất kỳ.

## Tóm lại

* RNN giữ một trạng thái cỡ cố định, 2 KB cho 512 chiều ở FP32, bất kể chuỗi dài bao nhiêu, nhưng các bước nối
  nhau thành một đường tới hạn dài bằng chuỗi; batch là chiều song song duy nhất của nó.
* Ma trận điểm tập trung tăng theo bình phương: 33.6 MB mỗi tầng mỗi đầu ở 4,096 token, 240 GB mỗi tầng ở
  100,000 token với 12 đầu.
* Chia khối với softmax trực tuyến giữ nguyên phép tính bậc hai nhưng bỏ được ma trận bậc hai.
* Trong một khối transformer, phần tập trung vượt phần tuyến tính khi độ dài chuỗi lớn hơn khoảng 6 lần chiều mô
  hình (với $d_{ff} = 4d$).
* Khi sinh token ở batch nhỏ, băng thông chia cho số byte trọng số đặt trần cho token mỗi giây; bộ nhớ đệm KV lớn
  lên tuyến tính theo ngữ cảnh và số người dùng.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kiến trúc mạng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch06-nn-architectures/), dựng từ chương
[*Network Architectures*](https://mlsysbook.ai/vol1/nn_architectures/nn_architectures.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_06_nn_arch.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.